# Zajęcia 3 — dane: pozyskanie, wersjonowanie, kontrakt i governance

Model jest funkcją danych. Dopóki nie wiemy dokładnie, **których** danych, nie da się powiedzieć, co właściwie zbudowaliśmy.

W tym notebooku:

1. pobieramy zbiór **ze zdalnego źródła** i zapisujemy jego tożsamość,
2. opisujemy go manifestem z sumą kontrolną,
3. wersjonujemy plik przez **DVC** i odtwarzamy go po skasowaniu,
4. definiujemy **kontrakt danych** i sprawdzamy, co się dzieje, gdy zostanie złamany,
5. szukamy wycieku etykiety,
6. spisujemy metryczkę zbioru.

Komórki opisane jako **zadanie** wypełniasz samodzielnie; komórka pod nimi zawiera rozwiązanie ukryte magią `%%skip True`.

## 1. Przygotowanie notebooka

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import textwrap
import time
from datetime import datetime, timezone
from pathlib import Path

import httpx
import pandas as pd
import pandera.pandas as pa
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "skip_kernel_extension.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BASE = REPO_ROOT / "artifacts" / "zajecia_03"
PROJECT_DIR = BASE / "projekt"
STORAGE_DIR = BASE / "magazyn"
REPORTS_DIR = BASE / "reports"
for folder in (PROJECT_DIR / "data", STORAGE_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
DATA_PATH = PROJECT_DIR / "data" / "iris.csv"
MANIFEST_PATH = PROJECT_DIR / "data" / "iris.manifest.json"


def write_file(path: Path, content: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")
    print(f"Zapisano: {path.name}")
    return path


def dvc(*arguments) -> subprocess.CompletedProcess:
    """Uruchamia DVC jako osobny proces w katalogu projektu."""
    result = subprocess.run(
        [sys.executable, "-m", "dvc", *arguments],
        cwd=PROJECT_DIR, capture_output=True, text=True,
    )
    output = (result.stdout + result.stderr).strip().splitlines()
    print(f"dvc {' '.join(arguments)} -> {output[-1][:110] if output else 'bez komunikatu'}")
    return result


print(f"Katalog projektu: {PROJECT_DIR.relative_to(REPO_ROOT)}")

%load_ext skip_kernel_extension

## 2. Pozyskanie zbioru ze zdalnego źródła

Dane pobieramy z konkretnego adresu, a nie z funkcji bibliotecznej. To jest sytuacja, którą spotkasz w projekcie: plik leży gdzieś na zewnątrz, ktoś inny nim zarządza i może go zmienić bez uprzedzenia.

**Zadanie:** napisz `fetch_dataset(url, destination)`, która pobiera plik, ponawia próbę przy błędzie sieci i **przy trwałym niepowodzeniu przechodzi na źródło zapasowe** zamiast przerywać pracę. Funkcja ma zwrócić informację, skąd faktycznie pochodzą dane.

In [ ]:
# Zadanie: napisz pobieranie danych ze zdalnego źródła.


In [ ]:
%%skip True
COLUMNS = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]


def _fallback_frame() -> pd.DataFrame:
    """Zrodlo zapasowe: ten sam zbior z biblioteki, sprowadzony do tych samych kolumn."""
    from sklearn.datasets import load_iris

    bundle = load_iris(as_frame=True)
    frame = bundle.frame.copy()
    frame.columns = COLUMNS[:4] + ["target"]
    frame["species"] = [bundle.target_names[value] for value in frame["target"]]
    return frame[COLUMNS]


def fetch_dataset(url: str, destination: Path, attempts: int = 3) -> dict:
    destination.parent.mkdir(parents=True, exist_ok=True)

    for attempt in range(1, attempts + 1):
        try:
            response = httpx.get(url, timeout=20.0, follow_redirects=True)
            response.raise_for_status()
            destination.write_bytes(response.content)
            return {"source": url, "origin": "zdalne", "attempts": attempt}
        except Exception as exc:
            print(f"  próba {attempt}/{attempts} nieudana: {type(exc).__name__}")
            if attempt < attempts:
                time.sleep(2**attempt)

    _fallback_frame().to_csv(destination, index=False)
    return {"source": "sklearn.datasets.load_iris", "origin": "zapasowe", "attempts": attempts}


provenance = fetch_dataset(DATA_URL, DATA_PATH)
raw = pd.read_csv(DATA_PATH)

print(f"\nŹródło: {provenance['origin']} ({provenance['source']})")
print(f"Wierszy: {len(raw)}, kolumn: {len(raw.columns)}")
display(raw.head(3))

## 3. Manifest zbioru

Plik danych bez opisu jest anonimowy. Manifest odpowiada na cztery pytania: **skąd**, **kiedy**, **co dokładnie** i **jak sprawdzić, że to wciąż to samo**.

Ostatnie pytanie rozstrzyga suma kontrolna. Nie liczymy jej „na wszelki wypadek" — liczymy ją po to, żeby przy każdym kolejnym uruchomieniu móc **odmówić pracy** na innych danych.

**Zadanie:** napisz `build_manifest(path, provenance)` tworzącą opis zbioru i zapisz go obok pliku danych.

In [ ]:
# Zadanie: zbuduj manifest zbioru.


In [ ]:
%%skip True
def sha256_of(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def build_manifest(path: Path, provenance: dict) -> dict:
    frame = pd.read_csv(path)
    manifest = {
        "name": path.stem,
        "source": provenance["source"],
        "origin": provenance["origin"],
        "retrieved_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "sha256": sha256_of(path),
        "size_bytes": path.stat().st_size,
        "rows": int(len(frame)),
        "columns": {name: str(dtype) for name, dtype in frame.dtypes.items()},
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


manifest = build_manifest(DATA_PATH, provenance)
print(json.dumps(manifest, indent=2))

## 4. Weryfikacja pliku względem manifestu

Manifest, którego nikt nie sprawdza, jest komentarzem. Weryfikacja ma być **pierwszym krokiem potoku** i ma go zatrzymywać.

**Zadanie:** napisz `verify_dataset(path, manifest_path)` zgłaszającą wyjątek, gdy plik nie odpowiada manifestowi. Sprawdź rozmiar, sumę kontrolną i zestaw kolumn.

In [ ]:
# Zadanie: napisz weryfikację pliku względem manifestu.


In [ ]:
%%skip True
class DatasetMismatch(RuntimeError):
    """Plik danych nie odpowiada temu, co opisuje manifest."""


def verify_dataset(path: Path, manifest_path: Path) -> dict:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    problems = []

    if not path.exists():
        raise DatasetMismatch(f"Brak pliku danych: {path.name}")

    actual_size = path.stat().st_size
    if actual_size != manifest["size_bytes"]:
        problems.append(f"rozmiar {actual_size} zamiast {manifest['size_bytes']}")

    actual_hash = sha256_of(path)
    if actual_hash != manifest["sha256"]:
        problems.append(f"suma kontrolna {actual_hash[:12]} zamiast {manifest['sha256'][:12]}")

    actual_columns = list(pd.read_csv(path, nrows=0).columns)
    if actual_columns != list(manifest["columns"]):
        problems.append(f"kolumny {actual_columns}")

    if problems:
        raise DatasetMismatch("Dane nie odpowiadają manifestowi: " + "; ".join(problems))

    print(f"Weryfikacja OK — {manifest['rows']} wierszy, suma {manifest['sha256'][:12]}...")
    return manifest


verify_dataset(DATA_PATH, MANIFEST_PATH)

## 5. Punkt kontrolny 1 — podmieniony plik danych

Zmieniamy jedną wartość w pliku. Nie zmienia się liczba wierszy, liczba kolumn ani nazwa pliku — zmienia się jedna liczba.

**Do opisania:** czy bez sumy kontrolnej ktokolwiek zauważyłby tę zmianę? Po czym?

In [ ]:
original_bytes = DATA_PATH.read_bytes()

tampered = pd.read_csv(DATA_PATH)
print(f"Wartość przed zmianą: {tampered.loc[0, 'sepal_length']}")
tampered.loc[0, "sepal_length"] = 5.2
tampered.to_csv(DATA_PATH, index=False)

try:
    verify_dataset(DATA_PATH, MANIFEST_PATH)
except DatasetMismatch as error:
    print(f"\nZATRZYMANO: {error}")
finally:
    DATA_PATH.write_bytes(original_bytes)

print("\nPo przywróceniu pliku:")
verify_dataset(DATA_PATH, MANIFEST_PATH)

## 6. Wersjonowanie danych — DVC

Suma kontrolna mówi, czy dane się zmieniły. Nie mówi, **jak wrócić** do poprzedniej wersji.

DVC rozwiązuje to tak: plik danych trafia do pamięci podręcznej i do zdalnego magazynu, a w repozytorium zostaje mały plik `.dvc` z sumą kontrolną. Git wersjonuje **wskaźnik**, DVC przechowuje **zawartość**.

```text
repozytorium Git          magazyn DVC
  data/iris.csv.dvc  ───▶  obiekt o sumie 9cc1c345...
  (kilkaset bajtów)        (właściwy plik)
```

W prawdziwym projekcie `dvc init` uruchamia się wewnątrz repozytorium Git — wtedy pliki `.dvc` są wersjonowane razem z kodem. Tutaj używamy `--no-scm`, żeby nie tworzyć zagnieżdżonego repozytorium wewnątrz repozytorium zajęć.

**Zadanie:** zainicjuj DVC, dodaj plik danych pod kontrolę, skonfiguruj magazyn lokalny i wyślij do niego dane.

In [ ]:
# Zadanie: obejmij dane kontrolą wersji.
# Wskazówki: dvc init --no-scm / dvc add / dvc remote add -d / dvc push


In [ ]:
%%skip True
if not (PROJECT_DIR / ".dvc").exists():
    dvc("init", "--no-scm", "-q")

dvc("add", "data/iris.csv")
dvc("remote", "add", "-d", "-f", "magazyn", str(STORAGE_DIR))
dvc("push")

pointer = PROJECT_DIR / "data" / "iris.csv.dvc"
print(f"\nWskaźnik {pointer.name} ({pointer.stat().st_size} B):")
print(pointer.read_text(encoding="utf-8"))
print(f"Plik danych: {DATA_PATH.stat().st_size} B")
print("\nTo właśnie wskaźnik trafia do repozytorium — nie dane.")

## 7. Punkt kontrolny 2 — utrata pliku danych

Kasujemy plik danych **oraz lokalną pamięć podręczną**. Zostaje tylko wskaźnik i magazyn — dokładnie ta sytuacja, w której znajdzie się osoba klonująca repozytorium.

**Do opisania:** czy odtworzony plik jest tym samym plikiem, czy tylko podobnym? Skąd to wiesz?

In [ ]:
DATA_PATH.unlink(missing_ok=True)
shutil.rmtree(PROJECT_DIR / ".dvc" / "cache", ignore_errors=True)
print(f"Plik danych istnieje: {DATA_PATH.exists()}")
print(f"Pamięć podręczna istnieje: {(PROJECT_DIR / '.dvc' / 'cache').exists()}\n")

dvc("pull")

print(f"\nPlik odtworzony: {DATA_PATH.exists()}")
verify_dataset(DATA_PATH, MANIFEST_PATH)
print("\nSuma kontrolna zgadza się z manifestem sprzed usunięcia —")
print("to jest ten sam plik co do bajtu, a nie jego rekonstrukcja.")

## 8. Kontrakt danych

Suma kontrolna sprawdza tożsamość pliku. Kontrakt sprawdza jego **sens**: czy wartości mieszczą się w dopuszczalnych zakresach, czy nie ma braków tam, gdzie ich być nie może, czy zbiór kategorii jest zamknięty.

Różnica jest zasadnicza: nowa dostawa danych **ma** mieć inną sumę kontrolną, ale **musi** spełniać ten sam kontrakt.

**Zadanie:** zdefiniuj schemat `pandera` dla zbioru. Uwzględnij typy, zakresy wartości, zamknięty zbiór gatunków, brak wartości pustych i tryb ścisły (żadnych dodatkowych kolumn).

In [ ]:
# Zadanie: zdefiniuj kontrakt danych.


In [ ]:
%%skip True
SPECIES = ["setosa", "versicolor", "virginica"]

SCHEMA = pa.DataFrameSchema(
    columns={
        "sepal_length": pa.Column(float, pa.Check.in_range(3.0, 9.0), nullable=False),
        "sepal_width": pa.Column(float, pa.Check.in_range(1.0, 5.0), nullable=False),
        "petal_length": pa.Column(float, pa.Check.in_range(0.5, 8.0), nullable=False),
        "petal_width": pa.Column(float, pa.Check.in_range(0.05, 3.0), nullable=False),
        "species": pa.Column(str, pa.Check.isin(SPECIES), nullable=False),
    },
    checks=[
        # Regula miedzykolumnowa: platek nigdy nie jest dluzszy od dzialki kielicha.
        pa.Check(lambda frame: frame["petal_length"] <= frame["sepal_length"], name="platek_krotszy"),
    ],
    strict=True,
    coerce=False,
)

print("Kontrakt obejmuje:")
for name, column in SCHEMA.columns.items():
    rules = ", ".join(str(check) for check in column.checks) or "tylko typ"
    print(f"  {name:14s} {column.dtype}  {rules}")
print(f"  + {len(SCHEMA.checks)} reguła międzykolumnowa")
print(f"  + tryb ścisły: {SCHEMA.strict}")

## 9. Walidacja i raport

Walidację uruchamiamy w trybie **leniwym** (`lazy=True`), żeby zobaczyć wszystkie naruszenia naraz, a nie tylko pierwsze. Wynik zapisujemy jako raport — to on trafi później do artefaktów przebiegu CI.

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
def validate(frame: pd.DataFrame, label: str) -> dict:
    try:
        SCHEMA.validate(frame, lazy=True)
        report = {"dataset": label, "valid": True, "violations": []}
    except pa.errors.SchemaErrors as errors:
        cases = errors.failure_cases
        report = {
            "dataset": label,
            "valid": False,
            "violations": (
                cases.groupby(["column", "check"], dropna=False)
                .size()
                .reset_index(name="count")
                .to_dict(orient="records")
            ),
        }
    (REPORTS_DIR / f"validation_{label}.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
    )
    return report


raw = pd.read_csv(DATA_PATH)
result = validate(raw, "produkcyjny")
print(f"Zbiór zgodny z kontraktem: {result['valid']}")

## 10. Punkt kontrolny 3 — złamany kontrakt

Symulujemy typową dostawę danych po zmianie w systemie źródłowym: kilka braków, jedna wartość poza zakresem, nowa kategoria i dodatkowa kolumna.

**Do opisania:** które z tych naruszeń wykryłby zwykły `read_csv` bez kontraktu? Które przeszłyby niezauważone aż do wyników modelu?

In [ ]:
broken = raw.copy()
broken.loc[0:4, "sepal_width"] = None
broken.loc[10, "petal_length"] = 87.0
broken.loc[11, "species"] = "Setosa"
broken["zrodlo_rekordu"] = "system_b"

report = validate(broken, "uszkodzony")
print(f"Zbiór zgodny z kontraktem: {report['valid']}\n")
display(pd.DataFrame(report["violations"]))

print(
    "Największe ryzyko niesie 'Setosa' zapisana wielką literą:\n"
    "nie jest brakiem ani wartością odstającą, więc przejdzie przez większość kontroli,\n"
    "a w modelu utworzy nową, prawie pustą klasę."
)

## 11. Wyciek etykiety

Najgroźniejsze dane to te, które wyglądają na świetne. Kolumna silnie skorelowana z celem — bo powstała **po** nim albo **z** niego — daje znakomity wynik w ocenie i bezużyteczny model w produkcji.

**Zadanie:** dopisz do zbioru kolumnę zależną od etykiety, porównaj jakość modelu z nią i bez niej, a następnie napisz `find_leaks(frame, target)` wskazującą podejrzane kolumny.

In [ ]:
# Zadanie: wykryj wyciek etykiety.


In [ ]:
%%skip True
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
rng = np.random.default_rng(42)

leaky = raw.copy()
codes = leaky["species"].astype("category").cat.codes
# Kolumna, ktora w rzeczywistosci powstaje PO poznaniu etykiety,
# na przyklad numer kolejny nadawany przy katalogowaniu okazu.
leaky["numer_katalogowy"] = codes.astype("int64") * 1000 + rng.integers(0, 999, size=len(leaky))

scores = {}
for label, columns in (("bez wycieku", FEATURES), ("z wyciekiem", FEATURES + ["numer_katalogowy"])):
    scores[label] = float(
        cross_val_score(
            RandomForestClassifier(n_estimators=200, random_state=42),
            leaky[columns], leaky["species"], cv=5, scoring="accuracy",
        ).mean()
    )
    print(f"{label:14s} trafność {scores[label]:.4f}")


def find_leaks(frame: pd.DataFrame, target: str, threshold: float = 0.9) -> pd.DataFrame:
    """Kolumna, ktora sama w sobie prawie idealnie przewiduje cel, jest podejrzana."""
    labels = frame[target].astype("category").cat.codes
    rows = []
    for column in frame.columns:
        if column == target or not pd.api.types.is_numeric_dtype(frame[column]):
            continue
        alone = float(
            cross_val_score(
                RandomForestClassifier(n_estimators=100, random_state=42),
                frame[[column]], labels, cv=5, scoring="accuracy",
            ).mean()
        )
        rows.append({"kolumna": column, "trafnosc_sama": round(alone, 4), "podejrzana": alone >= threshold})
    return pd.DataFrame(rows).sort_values("trafnosc_sama", ascending=False)


print()
display(find_leaks(leaky, "species"))
print(
    "Sam wysoki wynik nie jest dowodem wycieku — o tym rozstrzyga pytanie,\n"
    "czy ta kolumna będzie znana w momencie predykcji. Tu nie będzie."
)

## 12. Metryczka zbioru

Metryczka (*dataset card*) to dokument, który odpowiada na pytania zadawane przez osoby, których nie było przy powstawaniu zbioru: skąd te dane, kto za nie odpowiada, co wolno z nimi zrobić, jakie mają ograniczenia.

To jest element **governance**, a nie dokumentacji technicznej. Jego brak ujawnia się dopiero wtedy, gdy ktoś pyta, czy wolno wytrenować na tych danych model komercyjny.

**Zadanie:** wygeneruj `dataset_card.md` na podstawie manifestu i kontraktu. Uwzględnij pochodzenie, licencję, właściciela, zakres dopuszczalnego użycia, znane ograniczenia i kategorie danych osobowych.

In [ ]:
# Zadanie: wygeneruj metryczkę zbioru.


In [ ]:
%%skip True
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
columns_table = "\n".join(
    f"| `{name}` | {dtype} | {'cel' if name == 'species' else 'cecha'} |"
    for name, dtype in manifest["columns"].items()
)

write_file(
    REPORTS_DIR / "dataset_card.md",
    f"""
    # Metryczka zbioru — {manifest['name']}

    ## Tożsamość

    | Pole | Wartość |
    |---|---|
    | Źródło | {manifest['source']} |
    | Pobrano | {manifest['retrieved_at']} |
    | Suma kontrolna (SHA-256) | `{manifest['sha256']}` |
    | Rozmiar | {manifest['size_bytes']} B |
    | Liczba wierszy | {manifest['rows']} |

    ## Zawartość

    | Kolumna | Typ | Rola |
    |---|---|---|
{textwrap.indent(columns_table, '    ')}

    ## Pochodzenie i prawa

    - **Właściciel merytoryczny:** prowadzący przedmiot (zbiór dydaktyczny).
    - **Licencja:** zbiór publiczny, powszechnie stosowany w dydaktyce; przed użyciem
      komercyjnym należy sprawdzić warunki u dysponenta źródła.
    - **Dane osobowe:** brak. Zbiór opisuje pomiary roślin, nie osoby.
    - **Dopuszczalne użycie:** dydaktyka, testy potoku, przykłady w dokumentacji.

    ## Znane ograniczenia

    - Zbiór jest **mały i zrównoważony**, co nie odpowiada warunkom produkcyjnym;
      wyniki uzyskane na nim nie przenoszą się na zadania z klasami rzadkimi.
    - Pochodzi z jednego pomiaru w jednym miejscu — nie zawiera zmienności
      w czasie, więc **nie da się na nim badać dryfu** bez sztucznego wprowadzenia zmian.
    - Trafność powyżej 0.95 jest tu normą, nie osiągnięciem. Wysoki wynik na tym
      zbiorze nie świadczy o jakości rozwiązania.

    ## Kontrakt

    Zbiór musi spełniać schemat zdefiniowany w notebooku zajęć 3:
    typy kolumn, zakresy wartości, zamknięty zbiór gatunków, brak wartości pustych,
    brak kolumn dodatkowych oraz reguła `petal_length <= sepal_length`.

    ## Wersjonowanie

    Plik jest pod kontrolą DVC. W repozytorium znajduje się wskaźnik
    `data/iris.csv.dvc`; zawartość przechowuje magazyn skonfigurowany jako `magazyn`.
    Odtworzenie: `dvc pull`.
    """,
)

print()
print((REPORTS_DIR / "dataset_card.md").read_text(encoding="utf-8")[:900])

## 13. Poza notebookiem — dane w repozytorium

W projekcie zaliczeniowym DVC uruchamiasz **wewnątrz repozytorium Git**, bez `--no-scm`. Wtedy plik `.dvc` jest wersjonowany razem z kodem i historia danych staje się częścią historii projektu.

### Krok 1 — inicjalizacja

```bash
dvc init
git add .dvc .dvcignore
git commit -m "Wersjonowanie danych przez DVC"
```

### Krok 2 — objęcie danych kontrolą

```bash
dvc add data/surowe.csv
git add data/surowe.csv.dvc data/.gitignore
git commit -m "Dane wejściowe v1"
```

Zwróć uwagę, że `dvc add` **sam dopisuje** plik danych do `.gitignore`. Od tej pory Git widzi wyłącznie wskaźnik.

### Krok 3 — magazyn zdalny

Na zajęciach wystarczy katalog:

```bash
dvc remote add -d magazyn ../dane-magazyn
dvc push
```

W projekcie zespołowym w tym miejscu podaje się adres magazynu obiektowego (S3, Azure Blob, Google Cloud Storage) albo serwera SSH. Konfiguracja jest wersjonowana, **poświadczenia nie** — te trafiają do `.dvc/config.local`, który jest ignorowany przez Git.

### Krok 4 — sprawdzenie z drugiej strony

```bash
git clone <adres> sprawdzenie
cd sprawdzenie
dvc pull
```

Jeżeli po tych dwóch komendach masz komplet danych — wersjonowanie działa. Jeżeli musisz komukolwiek wysłać plik ręcznie — nie działa.

## 14. Podsumowanie

Zbiór ma jednoznaczną tożsamość (suma kontrolna), opis pochodzenia (manifest), wersjonowaną zawartość (DVC), sprawdzalny kontrakt (pandera) i metryczkę odpowiadającą na pytania o prawa i ograniczenia. Podmiana jednej wartości jest wykrywana, a skasowany plik odtwarza się co do bajtu.

Czego nadal brakuje:

- kontrakt sprawdzamy ręcznie, a nie jako pierwszy krok potoku (zajęcia 4);
- przetwarzanie cech żyje w notebooku, więc trening i predykcja mogą się rozjechać (zajęcia 4);
- nie widać, które etapy trzeba przeliczyć po zmianie danych (zajęcia 4).

### Zadanie do własnego projektu

1. Dane pobierane **skryptem**, nie ręcznie — z zapisanym adresem źródła.
2. Manifest z sumą kontrolną, datą pobrania i liczbą wierszy.
3. Weryfikacja manifestu jako **pierwszy krok** potoku, zatrzymująca pracę przy niezgodności.
4. Dane pod kontrolą DVC, magazyn skonfigurowany, `dvc push` wykonany.
5. Sprawdzone odtworzenie: świeży klon + `dvc pull` daje komplet danych.
6. Kontrakt danych obejmujący typy, zakresy, braki, zamknięte zbiory kategorii i co najmniej jedną regułę międzykolumnową.
7. Raport z walidacji zapisywany jako artefakt.
8. Analiza wycieku etykiety z uzasadnieniem dla każdej podejrzanej kolumny.
9. Metryczka zbioru z pochodzeniem, licencją, kategoriami danych osobowych i znanymi ograniczeniami.

**Kryterium ukończenia:** podmiana jednej wartości w pliku danych zatrzymuje potok, a `dvc pull` po skasowaniu katalogu `data/` przywraca zbiór o tej samej sumie kontrolnej.